# Demo threshold calibration and score parity (Kaggle)

Produces the files the Streamlit demo still needs, using **validation audio only** for the
threshold. The held-out test split is used only in Part C, to compare scores.

| Part | What it does | Required? | Rough time (T4 GPU) |
|---|---|---|---|
| **A** | Scores the clean validation split with C1 and A1, then writes a validation-derived threshold to `thresholds.json` | Yes | ~20-40 min |
| **B** | Re-encodes the validation audio with each codec (Opus/MP3/AAC), then checks the same threshold on it. Report only; the threshold is not changed | Recommended | ~40-60 min per codec, 6 codecs |
| **C** | Parity: runs the demo's own inference on 200 clean **test** clips and compares with the committed `results/preds_*_clean.csv` scores | Yes | ~5 min |

Times are estimates from local measurements (transcoding ~0.1 s/clip, bispectrum ~0.02 s/clip). Check them on the first run.

### Before running
1. Push `main` to GitHub. The notebook clones the repo; the demo files are only in local commits until pushed.
2. **Add data:** the `asvspoof-2024` dataset (the same one `build_manifest.py` used).
3. **Settings:** Accelerator **GPU T4**, Internet **On**.
4. Edit only the **CONFIG** cell.

### Running and resuming
- Use **Save Version → Save & Run All (Commit)** so it runs unattended and keeps `/kaggle/working` as the notebook output.
- Scores are cached per model and condition in `/kaggle/working/calibration/`. If a session dies, add the previous
  version's output as input, point `RESUME_FROM` at its `calibration` folder and run again. Finished conditions are skipped.
- Part B takes several hours in total. Setting `CODECS` to a few conditions per run is fine; the cache accumulates.

### GitHub upload setup
Create a fine-grained GitHub personal access token restricted to this repository with **Contents: Read and write**.
In Kaggle, open **Add-ons → Secrets**, add `GH_TOKEN`, and enable access for this notebook. Keep Internet on.
Never paste the token into a code cell. Part D commits `demo/thresholds.json` to `BRANCH` (default `main`)
after Part C passes. Set `UPLOAD_TO_GITHUB = False` to keep results local only.

### Afterwards
Part D prints the GitHub commit link, or reports that the file is already up to date.
`demo_thresholds.json` and `calibration/` remain in the Kaggle output for download; the CSV cache is not uploaded.
If branch protection blocks a direct commit, use your repository's normal review workflow instead.

References: [GitHub Contents API](https://docs.github.com/en/rest/repos/contents),
[Kaggle secrets](https://github.com/Kaggle/kaggle-cli/blob/main/docs/kernels.md).


In [ ]:
# ── CONFIG — edit only this cell ─────────────────────────────────────────────
REPO_URL    = "https://github.com/Remigaraki/TwoStream-Audio-Forensics.git"
UPLOAD_TO_GITHUB = True     # Part D: publish thresholds after parity passes
BRANCH      = "main"
DATA_ROOT   = "/kaggle/input/datasets/ulianazb/asvspoof-2024"   # contains flac_T/, flac_D/, ASVspoof5.*.metadata.txt

OPERATING_POINT = "eer"      # "eer" = equal error rates; "min_dcf" = paper's cost (bona fide wrongly flagged costs 1.9x)
MODELS      = ["C1", "A1"]
CODECS      = ["opus_16", "opus_32", "opus_64", "mp3_64", "mp3_128", "aac_128"]   # Part B; trim to split across sessions
RUN_PART_B  = True
PARITY_N    = 200            # Part C: number of clean test clips per model

RESUME_FROM = ""             # e.g. "/kaggle/input/<previous-version-output>/calibration" to reuse cached scores

# Paths (normally left alone)
REPO      = "/kaggle/tmp/TwoStream-Audio-Forensics"   # outside /kaggle/working: checkpoints are not re-saved as output
MANIFEST  = "/kaggle/working/manifest.csv"
CACHE_DIR = "/kaggle/working/calibration"
CODEC_VAL = "/kaggle/tmp/codec_val"                   # temporary; each codec folder is deleted once scored

In [ ]:
# ── Helpers: run a command, stream its output, stop on failure ───────────────
import os, shutil, subprocess, sys

ENV = {**os.environ, "PYTHONIOENCODING": "utf-8", "PYTHONUNBUFFERED": "1"}

def run(cmd, cwd=None):
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, cwd=cwd or REPO, env=ENV, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"command failed with exit code {p.returncode}")

In [ ]:
# ── 1. Clone the repo and fetch ONLY the two served checkpoints (Git LFS) ────
os.makedirs(os.path.dirname(REPO), exist_ok=True)
if not os.path.isdir(os.path.join(REPO, ".git")):
    shutil.rmtree(REPO, ignore_errors=True)   # leftovers of an interrupted clone
    # Skip LFS on clone so the other LFS checkpoints are not downloaded.
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO],
                   env={**ENV, "GIT_LFS_SKIP_SMUDGE": "1"}, check=True)
else:
    run(["git", "pull", "--ff-only"])

if subprocess.run(["git", "lfs", "version"], capture_output=True).returncode != 0:
    run(["bash", "-c", "apt-get -qq update && apt-get -qq install -y git-lfs"], cwd="/")
run(["git", "lfs", "install", "--local"])
run(["git", "lfs", "pull", "--include",
     "checkpoints/setup_c/best_c1_attention.pt,checkpoints/setup_a/best_a_ep13_eer0105_0713_0401.pt"])
run(["git", "log", "-1", "--oneline"])

In [ ]:
# ── 2. Dependencies: keep Kaggle's GPU torch; match the PCA's scikit-learn ───
# pca.pkl was pickled by scikit-learn 1.8.0. pydub is needed only because src/train.py imports it.
run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn==1.8.0", "pydub"])
run(["ffmpeg", "-hide_banner", "-version"])

In [ ]:
# ── 3. Environment + artifact check (hashes, LFS pointers, strict load) ──────
run([sys.executable, "-c", """
import sklearn, torch, torchaudio, soundfile
assert sklearn.__version__ == '1.8.0', sklearn.__version__
print('torch', torch.__version__, '| torchaudio', torchaudio.__version__, '| sklearn', sklearn.__version__,
      '| libsndfile', soundfile.__libsndfile_version__, '| cuda', torch.cuda.is_available())
from demo import inference as inf
for m in inf.MODELS:
    inf.load_model(m); print(m, 'verified and loaded')
"""])

In [ ]:
# ── 4. Evaluated manifest (must reproduce split hash ed4808bb0456de26) ───────
if not os.path.isfile(MANIFEST):
    run([sys.executable, "scripts/build_manifest.py", "--input_root", DATA_ROOT, "--output", MANIFEST])
run([sys.executable, "scripts/verify_manifest.py", "--manifest", MANIFEST])   # exits non-zero on any mismatch

os.makedirs(CACHE_DIR, exist_ok=True)
if RESUME_FROM:
    for f in os.listdir(RESUME_FROM):
        if f.endswith(".csv") and not os.path.exists(os.path.join(CACHE_DIR, f)):
            shutil.copy(os.path.join(RESUME_FROM, f), CACHE_DIR)
    print("cached score files:", sorted(os.listdir(CACHE_DIR)))

## Part A: validation threshold (clean audio)

Scores all 18,769 validation clips per model. It refuses to save unless the validation EER reproduces the checkpoint's
logged `val_eer` (±0.002), which confirms the setup matches training. Scores are cached, so re-running is instant.

In [ ]:
def calibrate(model, codec_root=None):
    cmd = [sys.executable, "demo/calibrate_threshold.py", "--model", model, "--manifest", MANIFEST,
           "--operating_point", OPERATING_POINT, "--cache_dir", CACHE_DIR, "--num_workers", "4"]
    run(cmd + (["--codec_root", codec_root] if codec_root else []))

for m in MODELS:
    calibrate(m)

## Part B: does the threshold hold up under compression? (validation audio)

For each codec: re-encode the validation clips, score them with each model at the threshold from Part A, then delete
the audio. The results are stored as `codec_check_val` in `thresholds.json`. Watch for a large gap between
`DCF` at the threshold and the `best achievable` value.

In [ ]:
def cached(model, cond):
    return any(f.startswith(f"val_{model}_") and f.endswith(f"_{cond}.csv") for f in os.listdir(CACHE_DIR))

if RUN_PART_B:
    for cond in CODECS:
        if all(cached(m, cond) for m in MODELS):
            print(f"[{cond}] already scored for {MODELS}, skipping"); continue
        run([sys.executable, "scripts/make_codec_testsets.py", "--manifest", MANIFEST,
             "--output_base", CODEC_VAL, "--split", "val", "--conditions", cond])
        for m in MODELS:
            calibrate(m, CODEC_VAL)          # clean + earlier codecs come from the cache
        shutil.rmtree(os.path.join(CODEC_VAL, cond), ignore_errors=True)
        print(f"[{cond}] done, audio deleted", flush=True)

## Results

In [ ]:
import json
t = json.load(open(os.path.join(REPO, "demo", "thresholds.json")))
for m, e in t.items():
    p = e.get("provenance", {})
    print(f"{m}: threshold={e['threshold']:.6f} status={e['status']} ({p.get('operating_point', '-')})")
    if p:
        print(f"    val EER={p['val_eer']:.4%} (checkpoint logged {p['checkpoint_logged_val_eer']:.4%}), "
              f"minDCF={p['val_min_dcf']:.4f}; at threshold: miss={p['val_miss_at_threshold']:.3%} "
              f"false alarm={p['val_false_alarm_at_threshold']:.3%}")
        for c, r in (p.get("codec_check_val") or {}).items():
            print(f"    {c:8s} miss={r['miss']:.3%} false alarm={r['false_alarm']:.3%} "
                  f"DCF={r['dcf_at_threshold']:.4f} (best {r['min_dcf']:.4f})")
shutil.copy(os.path.join(REPO, "demo", "thresholds.json"), os.path.join(os.path.dirname(CACHE_DIR), "demo_thresholds.json"))
print("\nSaved demo_thresholds.json next to calibration/ (download -> demo/thresholds.json). Part C runs next.")

## Part C: score parity with the evaluation pipeline (test audio, scores only)

Runs the **demo's own** inference path (bounded subprocess decode → preprocessing → CPU model) on the first `PARITY_N`
clean test clips and compares with the committed `results/preds_<model>_clean.csv` `score` column (tolerance 1e-4).
The test split is used only for this comparison, never for the threshold.

In [ ]:
import hashlib
from pathlib import Path

# Clear any successful marker from an earlier run before checking again.
PARITY_THRESHOLD_SHA256 = None
if not MODELS or PARITY_N <= 0:
    raise ValueError("Parity requires models and PARITY_N > 0")
parity_input = Path(REPO, "demo", "thresholds.json").read_bytes()
for m in MODELS:
    run([sys.executable, "demo/check_parity.py", "--model", m, "--manifest", MANIFEST, "--limit", str(PARITY_N)])
if Path(REPO, "demo", "thresholds.json").read_bytes() != parity_input:
    raise RuntimeError("Thresholds changed during parity; rerun Part C")
PARITY_THRESHOLD_SHA256 = hashlib.sha256(parity_input).hexdigest()


## Part D: upload verified thresholds to GitHub

Uses the `GH_TOKEN` Kaggle secret. Commits only `demo/thresholds.json` to `BRANCH`. Run Part C first in this session; rerun it if thresholds change.

In [ ]:
# Upload only the verified thresholds file; authentication stays in memory.
def upload_thresholds():
    import base64
    import hashlib
    import json
    import math
    import subprocess
    from pathlib import Path
    from urllib.error import HTTPError, URLError
    from urllib.parse import quote, urlparse
    from urllib.request import Request, urlopen

    content = Path(REPO, "demo", "thresholds.json").read_bytes()
    if hashlib.sha256(content).hexdigest() != globals().get("PARITY_THRESHOLD_SHA256"):
        raise RuntimeError("Run Part C successfully for the current thresholds before uploading.")
    thresholds = json.loads(content)
    if not MODELS:
        raise ValueError("MODELS must not be empty")
    for model in MODELS:
        entry = thresholds[model]
        if entry.get("status") != "validation" or not math.isfinite(entry["threshold"]):
            raise ValueError(f"{model} needs a finite, validation-calibrated threshold")

    parsed = urlparse(REPO_URL)
    parts = parsed.path.removesuffix(".git").strip("/").split("/")
    if parsed.scheme != "https" or parsed.netloc != "github.com" or len(parts) != 2:
        raise ValueError("REPO_URL must be https://github.com/OWNER/REPO.git")
    repo_slug = "/".join(quote(part, safe="") for part in parts)
    endpoint = f"https://api.github.com/repos/{repo_slug}/contents/demo/thresholds.json"
    base_sha = subprocess.check_output(
        ["git", "rev-parse", "HEAD:demo/thresholds.json"], cwd=REPO, text=True
    ).strip()
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GH_TOKEN")
        if not token or not token.strip():
            raise ValueError("Empty secret")
    except Exception:
        raise RuntimeError("Add GH_TOKEN in Kaggle Add-ons > Secrets and enable it for this notebook.") from None

    def api(method, url, payload=None):
        request = Request(url, method=method, headers={
            "Authorization": f"Bearer {token.strip()}",
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "Content-Type": "application/json",
            "User-Agent": "TwoStream-Kaggle-calibration",
        }, data=None if payload is None else json.dumps(payload).encode())
        try:
            with urlopen(request, timeout=60) as response:
                return json.load(response)
        except HTTPError as exc:
            # Do not dump requests, headers, or arbitrary server error bodies.
            raise RuntimeError(
                f"GitHub returned HTTP {exc.code}. Check token access, Contents write permission, "
                "branch protection, and whether the remote file changed. Local results are saved."
            ) from None
        except URLError:
            raise RuntimeError("GitHub connection failed. Check Kaggle Internet and retry; local results are saved.") from None

    remote = api("GET", endpoint + "?ref=" + quote(BRANCH, safe=""))
    if base64.b64decode(remote["content"]) == content:
        print(f"GitHub {BRANCH}: demo/thresholds.json is already up to date.")
        return
    if remote["sha"] != base_sha:
        raise RuntimeError("Thresholds changed on GitHub since this checkout. Review the remote changes before uploading.")
    result = api("PUT", endpoint, {
        "message": "Update demo thresholds from Kaggle validation calibration",
        "content": base64.b64encode(content).decode("ascii"),
        "sha": remote["sha"],
        "branch": BRANCH,
    })
    print("Uploaded demo/thresholds.json:", result["commit"]["html_url"])

if UPLOAD_TO_GITHUB:
    upload_thresholds()
else:
    print("GitHub upload disabled; results remain in Kaggle output.")
